# 06 — Work Item Orchestrator full loop (gated, expensive)

One end-to-end WIO cycle. The orchestrator is a single `ClaudeSDKClient` session that does:

1. **Step 1 (Phase 5, skill 10)** — Knowledge Store enrichment.
2. **Steps 2–4 (Phase 3)** — Builder -> Git -> QA cycle (cycle cap = 3 enforced by Pydantic + a Layer-2 escalation in this module).
3. **Step 5 (Phase 5, skill 11)** — Knowledge Store ingestion evaluation.

**Runtime: Claude-only.** WIO uses `claude_agent_sdk.ClaudeSDKClient` directly (not the runtime-agnostic seam) because the multi-turn stateful session has no Codex equivalent yet. `resolve_runtime("wio")` raises if `HSB_RUNTIME_WIO=codex` — we re-assert that hard-block below as a defence in depth.

**This notebook can spend real money and write to real Linear / GitHub.** Default state = explore only. Three gates must all be satisfied for the loop to run:

- `HSB_NOTEBOOK_RUN_LIVE=1`
- `HSB_NOTEBOOK_WIO_TASK_ID` set to a sandbox Linear LIN-ID
- `CLAUDE_CODE_OAUTH_TOKEN` set; `ANTHROPIC_API_KEY` and `OPENAI_API_KEY` UNSET

Inspection-only cells (system-prompt assembly, allow-list, G3 wiring, codex hard-block) run regardless and are the high-value half if you don't have credentials handy.

In [ ]:
import asyncio
import os
from pathlib import Path

from _helpers import (
    assert_g1_safe,
    ensure_src_on_path,
    gated,
    live_mode,
    selected_runtime,
)

ROOT = ensure_src_on_path()
from hsb.agents import work_item_orchestrator as wio

print("HSB_RUNTIME_WIO ->", selected_runtime("wio"), "(WIO is hard-blocked from codex)")

## WIO Codex hard-block

`resolve_runtime("wio")` must raise when `HSB_RUNTIME_WIO=codex`. Until WIO is ported off `ClaudeSDKClient`'s stateful session, this is the structural defence that keeps an operator from accidentally pointing it at Codex via env.

In [ ]:
from hsb.agents._sdk_options import resolve_runtime

prev = os.environ.get("HSB_RUNTIME_WIO")
os.environ["HSB_RUNTIME_WIO"] = "codex"
raised = False
try:
    resolve_runtime("wio")
except ValueError as e:
    raised = True
    assert "WIO" in str(e) or "wio" in str(e).lower()
assert raised, "WIO hard-block missing — resolve_runtime('wio') accepted codex"

if prev is None:
    os.environ.pop("HSB_RUNTIME_WIO", None)
else:
    os.environ["HSB_RUNTIME_WIO"] = prev
print("WIO hard-block: HSB_RUNTIME_WIO=codex raises ValueError as expected")

## System-prompt assembly — what the LLM actually receives

`assemble_system_prompt()` reads the 7 skill files (5 lifecycle skills + skills 10/11) and concatenates them with `# SKILL: <stem>` separators. We render the size + per-skill name list so you can spot a missing or duplicated skill at a glance.

In [ ]:
import os as _os
import re

# assemble_system_prompt() reads relative paths under skills/ — run from ROOT
# so nbconvert (which uses /tmp as cwd) and Jupyter Lab both resolve them.
_os.chdir(ROOT)
prompt = wio.assemble_system_prompt()
print(f"total prompt size: {len(prompt):,} chars (~{len(prompt) / 4:.0f} tokens)")
for m in re.finditer(r"^# SKILL: (.+?)$", prompt, re.MULTILINE):
    print(f"  - {m.group(1)}")
print("\nSKILL_FILES:")
for f in wio.SKILL_FILES:
    print("  ", f)

## G3 wiring — runtime backstop is called on every received message

`assert_no_task_dispatch` should appear in every receive loop. We grep the source for the call sites — if any loop loses the call, the runtime backstop disappears.

In [ ]:
src = Path(wio.__file__).read_text()
calls = src.count("assert_no_task_dispatch(")
print(f"assert_no_task_dispatch call sites: {calls}")
assert calls >= 3, (
    f"G3 runtime backstop missing or reduced (found {calls}, expected >= 3)"
)
assert '"Agent"' not in src, "WIO allow-list contains Agent"
# AgentDefinition is the sub-subagent surface. Check it's not actually
# imported (mentions in docstrings or comments are fine — they're documenting
# the absence).
ad_imports = [
    line
    for line in src.splitlines()
    if re.match(r"^\s*(from|import)\s+", line) and "AgentDefinition" in line
]
assert not ad_imports, f"WIO imports AgentDefinition: {ad_imports}"
# `agents=` as a keyword argument to ClaudeAgentOptions registers sub-agents.
# Strip comment-only lines before searching so the prose warning at line 232
# ('Do NOT register agents={}') doesn't false-positive.
non_comment = "\n".join(
    line for line in src.splitlines() if not line.lstrip().startswith("#")
)
kwarg_uses = re.findall(r"\bagents\s*=\s*[\[{]", non_comment)
assert not kwarg_uses, f"WIO passes agents= kwarg: {kwarg_uses}"
print("WIO: WORC-02 structural defences intact")

## In-process MCP tool wrappers (`@tool`)

Phase 2 agents are exposed via `create_sdk_mcp_server` rather than as sub-agents. The wrappers must each return the canonical `{"content":[{"type":"text","text":...}]}` envelope (Pitfall 4) — returning a Pydantic model directly silently fails the SDK serializer.

In [ ]:
for name in ["run_linear_tool", "run_builder_tool", "run_git_tool", "run_qa_tool"]:
    fn = getattr(wio, name, None)
    assert fn is not None, f"WIO missing {name}"
for name in ["run_linear_tool", "run_builder_tool", "run_git_tool", "run_qa_tool"]:
    block = src[src.index(f"async def {name}") : src.index(f"async def {name}") + 2000]
    assert '"content"' in block and '"text"' in block, (
        f"{name} missing canonical envelope"
    )
print("WIO: 4 @tool wrappers present, all return canonical content envelope")

## Live run (gated)

If all gates are satisfied, drive a single task end-to-end. Output is structured but verbose (skill prompts + every tool call); expect a few minutes wall-clock and several hundred KB of stdout.

**Before clicking run:** confirm the LIN-ID points to a sandbox project, not a production board.

In [ ]:
task_id = os.environ.get("HSB_NOTEBOOK_WIO_TASK_ID")
if not live_mode() or not task_id:
    print(
        gated(
            "WIO live run (set HSB_NOTEBOOK_WIO_TASK_ID=LIN-... and HSB_NOTEBOOK_RUN_LIVE=1)"
        )
    )
else:
    assert_g1_safe()
    print(f"Running WIO cycle for {task_id} (claude-only)…")
    asyncio.run(wio.run_orchestration_cycle(work_item_id=task_id))
    print("cycle complete — inspect Linear comments + GitHub PR for the artifacts")